# Four stocks: preprocessing and SDE-GAN experiment

Prepare the existing chronological CSV splits for the joint model described in
[sde_gan_spec.md](sde_gan_spec.md). Run all cells to audit, transform, window,
and save the data. Training and held-out evaluation are disabled by default. Select the
existing `sandbox/.venv` Python interpreter as the notebook kernel.
The source CSVs and [price audit](adjusted_price_histories.ipynb) remain the
source of the corrected, observed session calendar; no official calendar is
reconstructed here.

The generated scenarios are unconditional samples from the training period's
mixture of market conditions. The comparisons below are descriptive, not
forecasts or tests of calibrated portfolio risk. Overlapping observed windows
are not independent trajectories. Copula fitting is deferred.

In [ ]:
from pathlib import Path
from itertools import combinations
import json
import os
import sys
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = next(
    directory for directory in [Path.cwd(), *Path.cwd().parents]
    if (directory / "sandbox" / "4stocks_experiment" / "sde_gan_spec.md").is_file()
)
experiment_dir = root / "sandbox" / "4stocks_experiment"
data_dir = experiment_dir / "data"
results_dir = experiment_dir / "results" / "sde_gan"
STOCKS = ["KONGSBERG GRUPPEN", "FRONTLINE", "WILH Wilhelmsen B", "TOMRA SYSTEMS"]
SPLITS = ["train", "validation", "test"]
WINDOW_LENGTH = 20
STRIDE = 1
OBSERVATION_TIMES = np.arange(WINDOW_LENGTH)
TRAINING_SEEDS = [0, 1, 2]
SAMPLE_SEED = 100
N_PATHS = 1000
WEIGHTS = np.full(4, 0.25)

RUN_SMOKE = False
RUN_FULL_TRAINING = False
EVALUATE_TEST = False
# Set these only after reviewing the numerical checks and fitted-model sensitivity.
MODEL_NUMERICS_CHECKED = False
SOLVER_SENSITIVITY_CHECKED = False

print(f"Python: {sys.executable}")
print(f"Experiment: {experiment_dir}")

## Load and audit returns

Returns are fractions: `0.01` means 1%. Keep every calendar row, including
missing entries. Duplicate or unordered dates and overlapping splits are
errors. The exported calendar cannot reveal sessions missing for all stocks.

In [ ]:
def validate_frame(frame: pd.DataFrame, name: str) -> None:
    """Check the CSV schema and chronological row order without changing rows."""
    if len(frame.columns) != 4 or set(frame.columns) != set(STOCKS):
        raise ValueError(f"{name}: expected exactly the four stock columns")
    if frame.empty or frame.index.hasnans:
        raise ValueError(f"{name}: missing dates or empty split")
    if not frame.index.is_unique or not frame.index.is_monotonic_increasing:
        raise ValueError(f"{name}: dates must be unique and increasing")


def to_log_returns(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    """Transform simple returns while retaining zeros and missing entries."""
    invalid = frame.le(-1)
    if invalid.any().any():
        rows, cols = np.where(invalid.to_numpy())
        details = [(frame.index[i].strftime("%Y-%m-%d"), frame.columns[j])
                   for i, j in zip(rows[:10], cols[:10])]
        raise ValueError(f"{name}: simple return <= -1 at {details}")
    return np.log1p(frame)


def fit_scaler(training_logs: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Fit each stock on its nonmissing training rows, before windowing."""
    mean = training_logs.mean()
    std = training_logs.std(ddof=1)
    if not np.isfinite(mean).all() or not (np.isfinite(std) & std.gt(0)).all():
        raise ValueError("Training means must be finite and sample stds positive and finite")
    return mean, std


returns = {}
for split in SPLITS:
    frame = pd.read_csv(data_dir / f"{split}_returns.csv")
    frame["Date"] = pd.to_datetime(frame["Date"], format="%Y-%m-%d", errors="raise")
    frame = frame.set_index("Date")
    validate_frame(frame, split)
    returns[split] = frame.loc[:, STOCKS].astype(np.float64)
for left, right in zip(SPLITS, SPLITS[1:]):
    if returns[left].index[-1] >= returns[right].index[0]:
        raise ValueError(f"Overlapping or unordered {left}/{right} split dates")

split_audit = pd.DataFrame([
    {"split": split, "rows": len(frame), "first_date": frame.index[0],
     "last_date": frame.index[-1], "missing_entries": int(frame.isna().sum().sum())}
    for split, frame in returns.items()
]).set_index("split")
display(split_audit)
display(pd.concat({s: pd.DataFrame({"missing": f.isna().sum(), "zeros": f.eq(0).sum()})
                   for s, f in returns.items()}, names=["split", "stock"]))

## Log transformation and training scaler

Fit the mean and sample standard deviation (`ddof=1`) separately for each stock
using all its available training rows. Validation and test data never fit the
scaler. No clipping or per-window normalization is applied.

In [ ]:
log_returns = {s: to_log_returns(f, s) for s, f in returns.items()}
training_mean, training_std = fit_scaler(log_returns["train"])
standardized = {s: (f - training_mean) / training_std for s, f in log_returns.items()}
scaling_table = pd.DataFrame({
    "training_count": log_returns["train"].count(),
    "mean_log_return": training_mean,
    "sample_std_log_return": training_std,
})
display(scaling_table)
np.testing.assert_allclose(standardized["train"].mean(), 0, atol=1e-12)
np.testing.assert_allclose(standardized["train"].std(ddof=1), 1, atol=1e-12)
for split in SPLITS:
    assert log_returns[split].isna().equals(returns[split].isna())
    assert log_returns[split].eq(0).equals(returns[split].eq(0))

## Consecutive session windows

Inspect 20-row candidates before removing incomplete observations. A missing
entry invalidates every candidate containing it; rows on either side of a gap
are never joined. Build each split independently. Each array has axes
`(window, observation, stock)` with float32 values. Times `0, ..., 19` contain
20 actual return observations; no artificial initial zero is inserted.

In [ ]:
def extract_windows(frame: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """Keep complete consecutive windows and their matching calendar bounds."""
    values = frame.to_numpy(dtype=np.float64)
    paths, dates = [], []
    for start in range(0, len(frame) - WINDOW_LENGTH + 1, STRIDE):
        stop = start + WINDOW_LENGTH
        candidate = values[start:stop]
        if np.isfinite(candidate).all():
            paths.append(candidate)
            dates.append((frame.index[start], frame.index[stop - 1]))
    if not paths:
        raise ValueError("No complete 20-session windows in this split")
    with np.errstate(over="ignore"):
        array = np.ascontiguousarray(np.stack(paths), dtype=np.float32)
    if not np.isfinite(array).all():
        raise ValueError("Standardized windows exceed finite float32 range")
    metadata = pd.DataFrame(dates, columns=["start_date", "end_date"])
    metadata.index.name = "window_index"
    return array, metadata


windows, window_dates, counts = {}, {}, []
for split in SPLITS:
    try:
        windows[split], window_dates[split] = extract_windows(standardized[split])
    except ValueError as error:
        raise ValueError(f"{split}: {error}") from error
    candidates = max(0, (len(returns[split]) - WINDOW_LENGTH) // STRIDE + 1)
    counts.append({"split": split, "candidates": candidates, "retained": len(windows[split]),
                   "rejected": candidates - len(windows[split])})
    assert windows[split].shape[1:] == (20, 4)
    assert windows[split].dtype == np.float32 and windows[split].flags.c_contiguous
    assert window_dates[split]["start_date"].ge(returns[split].index[0]).all()
    assert window_dates[split]["end_date"].le(returns[split].index[-1]).all()
train_windows = windows["train"]
validation_windows = windows["validation"]
test_windows = windows["test"]
window_counts = pd.DataFrame(counts).set_index("split")
display(window_counts)
print({s: a.shape for s, a in windows.items()})

## Numerical and boundary checks

These small fixtures check gap handling, exact-length windows, invalid returns,
scaling, and inverse transforms. The current CSV baseline is 1,679 training,
709 validation, and 798 test windows; counts are reported rather than hardcoded
into the extraction logic.

In [ ]:
def expect_value_error(operation: Callable[[], object]) -> None:
    """Fail a fixture check if an invalid input is accepted."""
    try:
        operation()
    except ValueError:
        return
    raise AssertionError("Expected ValueError")


fixture = pd.DataFrame(
    np.arange(45 * 4, dtype=float).reshape(45, 4) / 1000,
    index=pd.date_range("2000-01-01", periods=45), columns=STOCKS,
)
fixture.iloc[22, 1] = np.nan
fixture_windows, fixture_dates = extract_windows(fixture)
assert len(fixture_windows) == 6  # starts 0, 1, 2 and 23, 24, 25
assert not ((fixture_dates.start_date <= fixture.index[22]) &
            (fixture_dates.end_date >= fixture.index[22])).any()
np.testing.assert_array_equal(fixture_windows[3], fixture.iloc[23:43].to_numpy(np.float32))
assert len(extract_windows(fixture.iloc[:20])[0]) == 1
assert len(extract_windows(fixture.iloc[23:43])[0]) == 1
expect_value_error(lambda: extract_windows(fixture.iloc[:19]))
expect_value_error(lambda: extract_windows(fixture * np.nan))
expect_value_error(lambda: extract_windows(pd.DataFrame(np.inf, index=fixture.index, columns=STOCKS)))
expect_value_error(lambda: to_log_returns(fixture.fillna(-1), "fixture"))
expect_value_error(lambda: to_log_returns(fixture.fillna(-1.1), "fixture"))
expect_value_error(lambda: fit_scaler(fixture.fillna(0) * 0))
expect_value_error(lambda: validate_frame(fixture.iloc[::-1], "fixture"))
expect_value_error(lambda: validate_frame(pd.concat([fixture, fixture]), "fixture"))
assert to_log_returns(fixture, "fixture").iloc[0, 0] == 0
assert np.isnan(to_log_returns(fixture, "fixture").iloc[22, 1])
fixture_mean, fixture_std = fit_scaler(fixture)
shifted_holdout = (fixture + 10 - fixture_mean) / fixture_std
assert shifted_holdout.mean().min() > 1  # not independently centered
np.testing.assert_array_equal(fixture_mean, fit_scaler(fixture)[0])
for split in SPLITS:
    # Reconstruct source windows directly by their dates, independently of scaling.
    source = np.stack([returns[split].loc[row.start_date:row.end_date].to_numpy()
                       for row in window_dates[split].itertuples()])
    recovered = np.expm1(windows[split].astype(np.float64) * training_std.to_numpy()
                         + training_mean.to_numpy())
    np.testing.assert_allclose(recovered, source, rtol=1e-6, atol=1e-7)
print("Preprocessing fixtures and source-window round trips passed.")

## Save prepared arrays and metadata

The NPZ contains only the three float32 model arrays. Window indices in the
CSV refer to their positions in those arrays. The JSON stores the training
scaler in the explicit stock order, plus calendar bounds and window settings.
Reload all three artifacts immediately to check the saved contract.

In [ ]:
preprocessing_metadata = {
    "stock_order": STOCKS, "transform": "log1p", "ddof": 1,
    "training_count": scaling_table.training_count.astype(int).tolist(),
    "training_mean": training_mean.tolist(), "training_std": training_std.tolist(),
    "window_length": WINDOW_LENGTH, "stride": STRIDE,
    "observation_times": OBSERVATION_TIMES.tolist(), "time_unit": "market session",
    "calendar": "Observed source calendar corrected in adjusted_price_histories.ipynb",
    "splits": {
        s: {"source": f"{s}_returns.csv", "first_date": f.index[0].strftime("%Y-%m-%d"),
            "last_date": f.index[-1].strftime("%Y-%m-%d"), "rows": len(f),
            **{k: int(v) for k, v in window_counts.loc[s].items()}}
        for s, f in returns.items()
    },
}
all_window_dates = pd.concat(window_dates, names=["split"]).reset_index()
data_dir.mkdir(exist_ok=True)
np.savez_compressed(data_dir / "prepared_windows.npz",
                    **{f"{s}_windows": a for s, a in windows.items()})
all_window_dates.to_csv(data_dir / "window_dates.csv", index=False, date_format="%Y-%m-%d")
(data_dir / "preprocessing.json").write_text(
    json.dumps(preprocessing_metadata, indent=2, allow_nan=False) + "\n"
)
with np.load(data_dir / "prepared_windows.npz", allow_pickle=False) as saved:
    assert set(saved.files) == {f"{s}_windows" for s in SPLITS}
    for split in SPLITS:
        assert saved[f"{split}_windows"].dtype == np.float32
        np.testing.assert_array_equal(saved[f"{split}_windows"], windows[split])
pd.testing.assert_frame_equal(
    pd.read_csv(data_dir / "window_dates.csv", parse_dates=["start_date", "end_date"]),
    all_window_dates,
)
assert json.loads((data_dir / "preprocessing.json").read_text()) == preprocessing_metadata
print(f"Saved and reloaded prepared arrays, dates, and scaler in {data_dir}")

## Inspect retained training coverage

Row-based standardized training statistics are zero mean and unit sample
variance. The retained-window distribution need not be: complete-case selection
and overlap change observation weights. Do not fit another scaler to correct
this difference. Held-out data are prepared above but not plotted before
training decisions are frozen.

In [ ]:
display(pd.DataFrame({
    "row_mean": standardized["train"].mean(),
    "row_sample_std": standardized["train"].std(ddof=1),
    "window_mean": train_windows.reshape(-1, 4).mean(axis=0),
    "window_sample_std": train_windows.reshape(-1, 4).std(axis=0, ddof=1),
}))
fig, axes = plt.subplots(3, 1, figsize=(13, 9), constrained_layout=True)
axes[0].imshow(returns["train"].isna().to_numpy().T, aspect="auto", interpolation="nearest",
               extent=(0, len(returns["train"]), 3.5, -0.5), cmap="Greys", vmin=0, vmax=1)
axes[0].set(yticks=range(4), yticklabels=STOCKS, xlabel="Training session row",
            title="Missing returns (black); calendar rows are preserved")
coverage = pd.Series(0, index=returns["train"].index)
coverage.loc[pd.DatetimeIndex(window_dates["train"].start_date)] = 1
coverage.resample("YS").sum().plot.bar(ax=axes[1])
axes[1].set(title="Retained training-window starts per year", ylabel="Windows", xlabel="Year")
axes[1].set_xticklabels([str(t.year) for t in coverage.resample("YS").sum().index], rotation=60)
for index in np.unique(np.linspace(0, len(train_windows) - 1, 3, dtype=int)):
    row = window_dates["train"].iloc[index]
    axes[2].plot(OBSERVATION_TIMES, train_windows[index, :, 0],
                 label=f"{row.start_date:%Y-%m-%d} to {row.end_date:%Y-%m-%d}")
axes[2].set(title=f"Example standardized training windows: {STOCKS[0]}",
            xlabel="Observation time (sessions)", ylabel="Standardized log return")
axes[2].legend()
plt.show()
plt.close(fig)

## Optional model setup and smoke run

Use the existing `sandbox/.venv` kernel for the JAX stack. Model imports occur
only when training is enabled. CPU selection must happen before JAX initializes;
restart the kernel if necessary. The notebook uses only the public model module.

Before setting `MODEL_NUMERICS_CHECKED`, review the model tests for known-SDE
behavior, full-penalty derivatives, generator gradients, and update isolation.
From this experiment directory, run the selected environment's Python with
`-m unittest test_sde_gan.TestInputsAndMetrics test_sde_gan.TestNumerics
test_sde_gan.TestGradientsAndAPI`. These checks are separate from this notebook.

Enable `RUN_SMOKE` and run the next cells first. The smoke fit uses 128 synthetic
training and 64 validation paths with covariance 1 on the diagonal and 0.5
off-diagonal, independent across time and paths. It is a numerical check,
not evidence of realistic market dynamics. Watch **Activity Monitor > Memory >
Memory Pressure** throughout the run; RSS alone does not measure system pressure.
Review post-compilation cycle time and peak RSS before enabling full training.

In [ ]:
if RUN_SMOKE or RUN_FULL_TRAINING:
    os.environ["JAX_PLATFORMS"] = "cpu"
    if str(experiment_dir) not in sys.path:
        sys.path.insert(0, str(experiment_dir))
    try:
        from sde_gan import SDEGAN, SDEGANConfig
    except ModuleNotFoundError as error:
        raise RuntimeError(
            f"Model dependencies unavailable in {sys.executable}; select sandbox/.venv "
            "and ensure sde_gan.py, training.py, and architecture.py are present."
        ) from error
else:
    print("Model imports and training disabled; preprocessing is complete.")

In [ ]:
def verify_checkpoint(model: Any, directory: Path) -> None:
    """Check finite sampling, explicit seeds, and inference restoration."""
    first = model.sample(n_paths=17, seed=SAMPLE_SEED)
    assert first.shape == (17, 20, 4) and np.isfinite(first).all()
    np.testing.assert_array_equal(first, model.sample(n_paths=17, seed=SAMPLE_SEED))
    assert not np.array_equal(first, model.sample(n_paths=17, seed=SAMPLE_SEED + 1))
    model.save(directory)
    restored = SDEGAN.load(directory)
    np.testing.assert_array_equal(first, restored.sample(n_paths=17, seed=SAMPLE_SEED))


if RUN_SMOKE:
    import threading
    from time import perf_counter
    import psutil

    rng = np.random.default_rng(2026)
    chol = np.linalg.cholesky(0.5 * np.eye(4) + 0.5 * np.ones((4, 4)))
    synthetic = (rng.normal(size=(192, 20, 4)) @ chol.T).astype(np.float32)
    smoke_model = SDEGAN(SDEGANConfig(generator_steps=50, validation_every=50))
    rss_records = []
    stop_monitor = threading.Event()
    process = psutil.Process()
    started = perf_counter()

    def record_rss() -> None:
        """Sample process memory during compilation, fitting, and validation."""
        while True:
            rss_records.append((perf_counter() - started, process.memory_info().rss / 2**20))
            if stop_monitor.wait(1):
                break

    monitor = threading.Thread(target=record_rss, daemon=True)
    monitor.start()
    smoke_passed = False
    try:
        smoke_history = pd.DataFrame(smoke_model.fit(synthetic[:128], synthetic[128:], seed=0))
        for record in smoke_model.history:
            assert all(np.isfinite(v) for v in record.values() if v is not None)
        assert set(smoke_history.loc[smoke_history.validation_mmd2.notna(), "step"]) == {0, 50}
        smoke_dir = results_dir / "smoke" / "seed_0"
        verify_checkpoint(smoke_model, smoke_dir)
        smoke_passed = True
    finally:
        stop_monitor.set()
        monitor.join()
    rss_frame = pd.DataFrame(rss_records, columns=["elapsed_seconds", "rss_mib"])
    rss_frame.to_csv(smoke_dir / "process_memory.csv", index=False)
    cycle_seconds = float(smoke_history.loc[smoke_history.step >= 2, "cycle_seconds"].median())
    display(pd.Series({"median_cycle_seconds_after_compilation": cycle_seconds,
                       "peak_sampled_rss_mib": rss_frame.rss_mib.max(),
                       "estimated_2000_cycle_seconds_excluding_validation": 2000 * cycle_seconds}))
    rss_frame.plot(x="elapsed_seconds", y="rss_mib", title="Smoke-run process RSS (sampled)")
    plt.show()
    plt.close("all")
    print("Smoke checks passed. Review system memory pressure before the full fits.")
else:
    print("Smoke fit disabled.")

## Optional full stock fits

Enable full training only after a successful smoke fit in this kernel and
reviewing its timings, memory pressure, and model numerical checks. Fits run
sequentially for seeds 0, 1, and 2, with 2,000 generator updates each. Every
fit starts fresh; test windows are never an argument. Validation MMD² and
checkpoint selection remain inside the model. Budgets do not prove convergence.

Each checkpoint receives its own preprocessing JSON so a restored generator
can be interpreted using its original stock labels and scaler. Re-running this
section replaces those seed directories' checkpoint files.

In [ ]:
if RUN_FULL_TRAINING:
    if not MODEL_NUMERICS_CHECKED or not globals().get("smoke_passed", False):
        raise RuntimeError("Review model numerical checks and complete RUN_SMOKE before full training")
    models, histories = {}, {}
    for seed in TRAINING_SEEDS:
        model = SDEGAN(SDEGANConfig())
        histories[seed] = pd.DataFrame(model.fit(train_windows, validation_windows, seed=seed))
        run_dir = results_dir / f"seed_{seed}"
        verify_checkpoint(model, run_dir)
        (run_dir / "preprocessing.json").write_text(
            json.dumps(preprocessing_metadata, indent=2, allow_nan=False) + "\n"
        )
        models[seed] = model
    display(pd.DataFrame([{"seed": seed, **model.run_metadata}
                          for seed, model in models.items()]))
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for seed, history in histories.items():
        axes[0, 0].plot(history.step, history.generator_loss, label=f"seed {seed} generator")
        axes[0, 0].plot(history.step, history.critic_loss, alpha=0.6, label=f"seed {seed} critic")
        axes[0, 1].plot(history.step, history.gradient_penalty, label=f"seed {seed}")
        for metric in ["generator_grad_norm", "critic_grad_norm", "input_grad_norm"]:
            axes[1, 0].plot(history.step, history[metric], label=f"{seed}: {metric}")
        scores = history.dropna(subset=["validation_mmd2"])
        axes[1, 1].plot(scores.step, scores.validation_mmd2, marker=".", label=f"seed {seed}")
    for ax, title in zip(axes.flat, ["Losses", "Gradient penalty", "Gradient norms", "Validation MMD²"]):
        ax.set(title=title, xlabel="Generator update")
        ax.legend(fontsize=7)
    plt.show()
    plt.close(fig)
else:
    print("Full stock training disabled.")

## Path diagnostics and portfolio calculation

The following helpers use simple returns in the saved stock order. Observed
windows use the original CSV values so inverse-transform rounding cannot erase
exact zero returns from their reported zero frequencies. Marginal and
contemporaneous statistics pool observations within windows, matching uniform
window sampling; overlapping dates therefore receive repeated weight. Absolute
return autocorrelation uses aligned within-path pairs, never joins the end of
one path to the next. Constant inputs produce unavailable correlations.

Downside events are returns at or below each stock's training-window 5%
quantile, held fixed for generated, validation, and test comparisons. Ties may
make the observed marginal frequency differ from 5%. Joint frequencies include
counts and denominators, and are finite-threshold co-loss measures, not claims
of asymptotic tail dependence. Spearman correlations use average ranks.

The portfolio holds 25% in each stock and rebalances each session. Its daily
simple return is `r @ weights`; its terminal return is `prod(1 + r_portfolio) - 1`.
Loss is the negative terminal return and reported VaR is its empirical 95%
quantile, without truncating negative values. Individual log returns are never
weighted to approximate exact portfolio returns.

In [ ]:
def source_simple_windows(split: str) -> np.ndarray:
    """Read retained observed windows from CSV values, preserving exact zeros."""
    return np.stack([
        returns[split].loc[row.start_date:row.end_date].to_numpy()
        for row in window_dates[split].itertuples()
    ])


def original_simple(paths: np.ndarray, metadata: dict) -> np.ndarray:
    """Invert standardized samples using the scaler paired with their generator."""
    if metadata["stock_order"] != STOCKS:
        raise ValueError("Checkpoint stock order differs from this notebook")
    with np.errstate(over="raise", invalid="raise"):
        simple = np.expm1(paths.astype(np.float64) * np.asarray(metadata["training_std"])
                          + np.asarray(metadata["training_mean"]))
    if not np.isfinite(simple).all():
        raise ValueError("Nonfinite inverse-transformed returns")
    return simple


def terminal_returns(simple: np.ndarray) -> np.ndarray:
    """Compound all 20 daily returns of the rebalanced portfolio."""
    daily = simple @ WEIGHTS
    result = np.prod(1 + daily, axis=1) - 1
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite terminal portfolio returns")
    return result


def path_statistics(simple: np.ndarray, thresholds: np.ndarray) -> dict[str, pd.DataFrame]:
    """Compute descriptive marginals, dependence, temporal, and portfolio summaries."""
    flat = pd.DataFrame(simple.reshape(-1, 4), columns=STOCKS)
    marginals = pd.DataFrame({"mean": flat.mean(), "sample_std": flat.std(ddof=1),
                              "zero_fraction": flat.eq(0).mean()})
    for q in [0.01, 0.05, 0.5, 0.95, 0.99]:
        marginals[f"q{q:.2f}"] = flat.quantile(q)
    acf = pd.DataFrame(index=range(1, 6), columns=STOCKS, dtype=float)
    for lag in acf.index:
        left = pd.DataFrame(np.abs(simple[:, :-lag]).reshape(-1, 4), columns=STOCKS)
        right = pd.DataFrame(np.abs(simple[:, lag:]).reshape(-1, 4), columns=STOCKS)
        acf.loc[lag] = left.corrwith(right)
    acf.index.name = "lag"
    downside = flat.to_numpy() <= thresholds
    groups = [*combinations(range(4), 2), (0, 1, 2, 3)]
    tails = pd.DataFrame([
        {"stocks": " / ".join(STOCKS[i] for i in group),
         "count": int(downside[:, group].all(axis=1).sum()), "denominator": len(flat),
         "frequency": downside[:, group].all(axis=1).mean()}
        for group in groups
    ]).set_index("stocks")
    terminal = terminal_returns(simple)
    portfolio = pd.DataFrame([{
        "paths": len(simple), "mean_terminal_return": terminal.mean(),
        "terminal_q05": np.quantile(terminal, 0.05),
        "terminal_q50": np.quantile(terminal, 0.5),
        "terminal_q95": np.quantile(terminal, 0.95),
        "loss_var_95": np.quantile(-terminal, 0.95),
    }])
    return {"marginals": marginals, "pearson": flat.corr(),
            "spearman": flat.corr(method="spearman"), "absolute_acf": acf,
            "joint_downside": tails, "portfolio": portfolio}


# Hand-calculated portfolio fixture: a 10% move in one stock gives 2.5% portfolio return.
portfolio_fixture = np.zeros((1, 20, 4))
portfolio_fixture[0, 0, 0] = 0.10
np.testing.assert_allclose(terminal_returns(portfolio_fixture), [0.025])
portfolio_fixture[0, 1, 1] = -0.10
np.testing.assert_allclose(terminal_returns(portfolio_fixture), [1.025 * 0.975 - 1])

## Optional validation comparisons

Before interpreting stock fits, compare solver steps 0.25 and 0.125 with shared
Brownian realizations as required by the spec. Keep that numerical verification
in the model checks; set `SOLVER_SENSITIVITY_CHECKED` only after reviewing it for
the fitted stock models. This section then samples each selected generator and
writes tables for every seed. Seed 0 supplies the representative plots; seeds
are not chosen using test performance. A continuous generator does not explicitly
represent the observed point mass at zero; compare zero frequencies with care.

In [ ]:
if RUN_FULL_TRAINING and SOLVER_SENSITIVITY_CHECKED:
    observed_simple = {s: source_simple_windows(s)
                       for s in ["train", "validation"]}
    downside_thresholds = np.quantile(observed_simple["train"].reshape(-1, 4), 0.05, axis=0)
    generated_simple = {}
    validation_summaries = {}
    for seed, model in models.items():
        run_dir = results_dir / f"seed_{seed}"
        paired_metadata = json.loads((run_dir / "preprocessing.json").read_text())
        generated = model.sample(n_paths=N_PATHS, seed=SAMPLE_SEED)
        assert generated.shape == (N_PATHS, 20, 4) and np.isfinite(generated).all()
        generated_simple[seed] = original_simple(generated, paired_metadata)
        np.savez_compressed(run_dir / "generated_paths.npz", standardized_log_returns=generated)
        datasets = {**observed_simple, "generated": generated_simple[seed]}
        validation_summaries[seed] = {name: path_statistics(a, downside_thresholds)
                                      for name, a in datasets.items()}
        for metric in validation_summaries[seed]["train"]:
            table = pd.concat({name: stats[metric] for name, stats in validation_summaries[seed].items()},
                              names=["dataset"])
            table.to_csv(run_dir / f"validation_{metric}.csv")
            print(f"Seed {seed}: {metric}")
            display(table)
        (run_dir / "evaluation.json").write_text(json.dumps({
            "sample_seed": SAMPLE_SEED, "n_paths": N_PATHS, "portfolio_weights": WEIGHTS.tolist(),
            "downside_training_thresholds": downside_thresholds.tolist(),
            "var_confidence": 0.95, "rebalancing": "each session",
        }, indent=2, allow_nan=False) + "\n")
    representative = {**observed_simple, "generated": generated_simple[0]}
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, stock in zip(axes.flat, range(4)):
        for name, array in representative.items():
            values = np.sort(array[:, :, stock].ravel())
            ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
        ax.set(title=STOCKS[stock], xlabel="Simple daily return", ylabel="Empirical CDF")
        ax.legend()
    plt.show()
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for name, array in representative.items():
        values = np.sort(terminal_returns(array))
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
    ax.set(title="20-session equal-weight rebalanced portfolio", xlabel="Terminal simple return",
           ylabel="Empirical CDF")
    ax.legend()
    plt.show()
    plt.close(fig)
else:
    print("Validation comparisons require full fits and reviewed solver sensitivity.")

## Optional final test comparisons

Freeze model choices, preprocessing, thresholds, weights, and evaluation formulas
before setting `EVALUATE_TEST`. This cell loads the saved checkpoints and their
paired scalers and evaluation settings, so it can also be run in a fresh kernel
without refitting. It checks that the prepared data match those settings. All
three seeds are reported; none is selected on the test set. Overlap and small
numbers of rare losses limit interpretation of the resulting VaR comparisons.

In [ ]:
if EVALUATE_TEST:
    if not SOLVER_SENSITIVITY_CHECKED:
        raise RuntimeError("Review fitted-model solver sensitivity before test interpretation")
    os.environ["JAX_PLATFORMS"] = "cpu"
    if str(experiment_dir) not in sys.path:
        sys.path.insert(0, str(experiment_dir))
    from sde_gan import SDEGAN, SDEGANConfig

    for seed in TRAINING_SEEDS:
        run_dir = results_dir / f"seed_{seed}"
        paired_metadata = json.loads((run_dir / "preprocessing.json").read_text())
        if paired_metadata != preprocessing_metadata:
            raise ValueError("Prepared data settings differ from this checkpoint's preprocessing")
        settings = json.loads((run_dir / "evaluation.json").read_text())
        if settings["portfolio_weights"] != WEIGHTS.tolist():
            raise ValueError("Notebook portfolio weights differ from frozen evaluation settings")
        restored = SDEGAN.load(run_dir)
        fake = original_simple(restored.sample(n_paths=settings["n_paths"],
                                               seed=settings["sample_seed"]), paired_metadata)
        real = source_simple_windows("test")
        summaries = {name: path_statistics(array, np.asarray(settings["downside_training_thresholds"]))
                     for name, array in {"test": real, "generated": fake}.items()}
        for metric in summaries["test"]:
            table = pd.concat({name: stats[metric] for name, stats in summaries.items()}, names=["dataset"])
            table.to_csv(run_dir / f"test_{metric}.csv")
            print(f"Seed {seed}: test {metric}")
            display(table)
else:
    print("Test evaluation disabled. Freeze experiment choices before enabling it.")